### Text Feature Engineering Assignment (Real-world Dataset)

praise@0407

In [6]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os

# ---------- CONFIG ----------
#BASE_URL = "https://www.amazon.com/product-reviews/0578973839/?reviewerType=all_reviews"
#BASE_URL = "https://www.amazon.com/product-reviews/B08VL5K5ZX/?reviewerType=all_reviews"
BASE_URL = "https://www.amazon.com/product-reviews/1098102932//?reviewerType=all_reviews"
OUTPUT_FILE = "amazon_reviews.csv"
TARGET_REVIEWS = 150

# ---------- CHROME SETUP ----------
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-gpu")
options.add_argument("--disable-extensions")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--remote-debugging-port=9222")

# Use a dedicated local profile folder and ensure it is not already in use
profile_path = r"C:\amazon_profile_chrome"
os.makedirs(profile_path, exist_ok=True)
options.add_argument(fr"--user-data-dir={profile_path}")

# Real user-agent
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
)

driver = webdriver.Chrome(options=options)
driver.implicitly_wait(10)
wait = WebDriverWait(driver, 30)

# ---------- HUMAN SCROLL ----------
def human_scroll():
    for _ in range(6):
        driver.execute_script("window.scrollBy(0, 800)")
        time.sleep(2)

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
    time.sleep(3)


# ---------- LOGIN ----------
def ensure_logged_in():
    driver.get(BASE_URL)
    time.sleep(3)

    print("\n🔐 STEP:")
    print("1. Login manually")
    print("2. Open reviews page")
    print("3. Scroll slowly until reviews fully load\n")

    input("👉 After doing above, press ENTER to continue...")

    # 🔥 Force reload AFTER manual interaction
    driver.get(BASE_URL)
    time.sleep(5)
    human_scroll()

    print("✅ Proceeding to scraping...\n")


# ---------- LOAD EXISTING DATA ----------
data = []
unique_reviews = set()

if os.path.exists(OUTPUT_FILE):
    try:
        if os.path.getsize(OUTPUT_FILE) > 0:
            df_existing = pd.read_csv(OUTPUT_FILE)
            data = df_existing.to_dict("records")

            if "review_text" in df_existing.columns:
                unique_reviews = set(df_existing["review_text"].astype(str).apply(hash))

            print(f"📂 Loaded {len(data)} existing reviews")
    except:
        print("⚠️ CSV issue → starting fresh")

# ---------- START ----------
ensure_logged_in()

# ---------- HELPERS ----------
def get_review_text(review):
    selectors = [
        './/span[@data-hook="review-body"]',
        './/span[@data-hook="review-body"]//span',
        './/div[@data-hook="review-collapsed"]',
        './/span[contains(@class,"review-text-content")]',
        './/span[contains(@class,"review-text-content")]//span',
        './/div[contains(@class,"review-text")]',
        './/div[contains(@id,"review") and contains(@class,"review")]',
    ]
    for selector in selectors:
        try:
            element = review.find_element(By.XPATH, selector)
            text = element.text.strip()
            if text:
                return text
        except:
            continue
    # Fallback: gather any visible text inside the review block
    try:
        text = review.text.strip()
        return text if text else None
    except:
        return None


def get_reviewer_name(review):
    selectors = [
        './/span[@class="a-profile-name"]',
        './/a[contains(@class,"a-profile-name")]',
        './/span[contains(@class,"a-profile-name")]',
        './/span[contains(@class,"profile-name")]',
    ]
    for selector in selectors:
        try:
            text = review.find_element(By.XPATH, selector).text.strip()
            if text:
                return text
        except:
            continue
    return None


def click_load_more_reviews():
    elements = driver.find_elements(
        By.XPATH,
        "//button[contains(.,'Show 10 more reviews') or contains(.,'Show more reviews') or contains(.,'Load more reviews') or contains(.,'See more reviews') or contains(.,'Load more') or contains(.,'Read more')]"
        + " | //a[contains(.,'Show 10 more reviews') or contains(.,'Show more reviews') or contains(.,'Load more reviews') or contains(.,'See more reviews') or contains(.,'Load more') or contains(.,'Read more')]"
        + " | //span[contains(.,'Show 10 more reviews') or contains(.,'Show more reviews') or contains(.,'Load more reviews') or contains(.,'See more reviews') or contains(.,'Load more') or contains(.,'Read more')]"
    )
    clicked = False
    for element in elements:
        try:
            driver.execute_script("arguments[0].scrollIntoView(true);", element)
            time.sleep(1)
            element.click()
            clicked = True
            time.sleep(3)
        except:
            continue
    return clicked

# ---------- SCRAPING ----------
for page in range(1, 2):

    if len(unique_reviews) >= TARGET_REVIEWS:
        break

    url = f"{BASE_URL}&pageNumber={page}"
    print(f"\n📄 Scraping page {page}")

    driver.get(url)
    time.sleep(5)

    # Human-like scroll
    human_scroll()

    # Click any "show more" / "load more" buttons if present
    for _ in range(3):
        if not click_load_more_reviews():
            break

    # Wait for reviews
    try:
        reviews = wait.until(
            EC.presence_of_all_elements_located((By.XPATH, '//div[@data-hook="review"] | //div[contains(@id,"customer_review")]'))
        )
    except Exception as e:
        print(f"🔄 Retry after refresh... {type(e).__name__}: {e}")
        driver.refresh()
        time.sleep(5)
        human_scroll()
        reviews = driver.find_elements(By.XPATH, '//div[@data-hook="review"] | //div[contains(@id,"customer_review")]')

    print(f"Found {len(reviews)} reviews")

    if len(reviews) == 0:
        print("⚠️ Skipping page (blocked or not loaded)")
        continue

    if page == 1:
        first_text = get_review_text(reviews[0])
        print("✅ First review preview:", first_text[:300] if first_text else "<no text extracted>")

    for review in reviews:
        if len(unique_reviews) >= TARGET_REVIEWS:
            break

        try:
            name = get_reviewer_name(review) or "Unknown"
            text = get_review_text(review)
            if not text:
                print("⚠️ Skipping review because text could not be extracted")
                continue


            key = hash(text)

            if key not in unique_reviews:
                unique_reviews.add(key)
                data.append({
                    "reviewer_name": name,
                    "review_text": text
                })

        except Exception as e:
            print(f"⚠️ Review skipped due to exception: {type(e).__name__}")
            continue

    print(f"✅ Total reviews collected: {len(unique_reviews)}")

    # Save progress
    if len(data) > 0:
        pd.DataFrame(data).to_csv(OUTPUT_FILE, index=False)

    time.sleep(3)

driver.quit()

print(f"\n🎯 Final saved reviews: {len(data)}")


🔐 STEP:
1. Login manually
2. Open reviews page
3. Scroll slowly until reviews fully load

✅ Proceeding to scraping...


📄 Scraping page 1
Found 40 reviews
✅ First review preview: This book breaks down the often-intimidating world of data science into something approachable. The explanations are clear, and the examples are practical, making it perfect for beginners or anyone brushing up on their math skills. It’s a great resource for tackling the math side of data science wit
✅ Total reviews collected: 40

🎯 Final saved reviews: 40
